# Test Bottom-Up Tree Construction

This notebook exercises the bottom-up semantic tree builder in `build_llm_bottom_up_tree.py`.

What it does:
- loads a corpus that already contains embeddings
- builds leaf nodes
- asks an LLM to summarize clusters while building parent nodes bottom-up
- inspects the resulting tree
- optionally saves the tree to `trees/`

Before running the build cells, make sure the required API key for your selected LLM backend is available in the environment.


In [2]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path
from types import SimpleNamespace


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists() and (current / "trees").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tree_construction.build_llm_bottom_up_tree import (
    ClusterSummarizer,
    build_bottom_up_tree,
    build_leaf_nodes,
    count_leaves,
    count_nodes,
    load_source_records,
    max_branching,
)
from utils import setup_logger

REPO_ROOT


c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.

WindowsPath('C:/Users/jmgil/Desktop/TESE/LATTICE/llm-guided-retrieval')

In [3]:
# Adjust these values for the corpus and model you want to test.
args = SimpleNamespace(
    input=str(REPO_ROOT / "src" / "tree_construction" / "codigo_civil_chunks_sample.json"),
    dataset="PT",
    subset="codigo_civil_sample",
    id_field="chunk_id",
    text_field="text",
    embedding_field="embedding",
    max_leaves=64,
    max_children=6,
    cluster_linkage="average",
    prompt_child_char_limit=700,
    summary_cache=str(REPO_ROOT / "results" / "tree_construction" / "codigo_civil_summary_cache.json"),
    llm_api_backend="openai",  # one of: openai, genai, vllm
    llm="gpt-4.1",
    llm_max_concurrent_calls=4,
    llm_api_timeout=120,
    llm_api_max_retries=4,
    llm_api_staggering_delay=0.05,
    vllm_base_url="http://localhost:8000/v1",
)

LOG_DIR = REPO_ROOT / "results" / "tree_construction"
LOG_DIR.mkdir(parents=True, exist_ok=True)
logger = setup_logger("bottom_up_tree_notebook", str(LOG_DIR / "bottom_up_tree_notebook.log"))

args


Log file already exists: C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\bottom_up_tree_notebook.log, appending to it.


namespace(input='C:\\Users\\jmgil\\Desktop\\TESE\\LATTICE\\llm-guided-retrieval\\src\\tree_construction\\codigo_civil_chunks_sample.json',
          dataset='PT',
          subset='codigo_civil_sample',
          id_field='chunk_id',
          text_field='text',
          embedding_field='embedding',
          max_leaves=64,
          max_children=6,
          cluster_linkage='average',
          prompt_child_char_limit=700,
          summary_cache='C:\\Users\\jmgil\\Desktop\\TESE\\LATTICE\\llm-guided-retrieval\\results\\tree_construction\\codigo_civil_summary_cache.json',
          llm_api_backend='openai',
          llm='gpt-4.1',
          llm_max_concurrent_calls=4,
          llm_api_timeout=120,
          llm_api_max_retries=4,
          llm_api_staggering_delay=0.05,
          vllm_base_url='http://localhost:8000/v1')

In [4]:
records = load_source_records(args, logger)
leaf_nodes = build_leaf_nodes(records, args, logger)

print(f"Loaded {len(records)} raw records")
print(f"Prepared {len(leaf_nodes)} leaf nodes")
print("First leaf preview:")
print(leaf_nodes[0].id)
print(leaf_nodes[0].desc[:400])


2026-04-29 16:49:22,069 - bottom_up_tree_notebook - INFO - Loading records from C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\src\tree_construction\codigo_civil_chunks_sample.json
2026-04-29 16:49:22,178 - bottom_up_tree_notebook - INFO - Prepared 64 leaf nodes


Loaded 64 raw records
Prepared 64 leaf nodes
First leaf preview:
b4ba2281eded1704ad27dc7defc4dea3c7009cd7
CC Art. 1.º (Aprovação do Código Civil)


In [5]:
summarizer = ClusterSummarizer(args, logger)
root = build_bottom_up_tree(leaf_nodes, args, summarizer, logger)
tree_dict = root.to_tree_dict()

print("Root description:")
print(tree_dict["desc"])
print()
print(f"Total nodes: {count_nodes(tree_dict)}")
print(f"Total leaves: {count_leaves(tree_dict)}")
print(f"Max branching: {max_branching(tree_dict)}")


2026-04-29 16:49:22,202 - bottom_up_tree_notebook - INFO - Initialized client for model: gpt-4.1
2026-04-29 16:49:22,578 - bottom_up_tree_notebook - INFO - Initialized OpenAI Responses client with model: gpt-4.1
2026-04-29 16:49:22,579 - bottom_up_tree_notebook - INFO - Building parent level 0 from 64 nodes
2026-04-29 16:49:22,728 - bottom_up_tree_notebook - INFO - Level 0 produced 19 parent nodes (max fanout 6)
2026-04-29 16:49:22,729 - bottom_up_tree_notebook - INFO - Building parent level 1 from 19 nodes
2026-04-29 16:49:22,737 - bottom_up_tree_notebook - INFO - Level 1 produced 9 parent nodes (max fanout 6)
2026-04-29 16:49:22,738 - bottom_up_tree_notebook - INFO - Building parent level 2 from 9 nodes
2026-04-29 16:49:22,741 - bottom_up_tree_notebook - INFO - Level 2 produced 4 parent nodes (max fanout 6)
2026-04-29 16:49:22,743 - bottom_up_tree_notebook - INFO - Building root from 4 top-level nodes


Root description:
ROOT Node: This subtree covers foundational, transitional, and historical aspects of the Portuguese Civil Code, including its approval, temporal application, and revocation of prior laws. It addresses general legal principles, sources of law, legal incapacities, property and credit rules, special lease and partnership regimes, guardianship and conservatorship frameworks, and the legislative evolution and constitutional review of specific articles. Use this node for questions about the structure and application of the Civil Code, transitional legal provisions, historical regulations, and the impact of legislative changes.. Key topics: transitional provisions; legal incapacities; universal and family partnerships; guardianship and conservatorship; revocation of Civil Code articles; historical legal frameworks; property and credit rules; special lease regimes. 4 direct children. 64 leaf passages

Total nodes: 97
Total leaves: 64
Max branching: 6


In [6]:
# Inspect the first level of the tree to see what summaries the LLM produced.
for idx, child in enumerate(tree_dict.get("child") or []):
    print(f"[{idx}] {child['id']}")
    print(child["desc"][:800])
    print("-" * 120)


[0] level:2:cluster:0000:9c02de3eb5
Portuguese Civil Code: General and Transitional Rules. This subtree addresses foundational and transitional aspects of the Portuguese Civil Code, including its approval, temporal application, and revocation of prior laws; general legal principles and sources; legal incapacities and their consequences; credit privileges and asset regimes; special rules for Lisbon/Porto leases and agricultural partnerships; and procedures for challenging legitimacy and parentage. Use this node for questions about the Code’s structure, transitional provisions, incapacity criteria, property and credit rules, lease exceptions, and family law procedures.. Key topics: transitional provisions; legal incapacities (interdições); credit privileges and mortgages; Lisbon/Porto lease laws; agricultural partnerships; gen
------------------------------------------------------------------------------------------------------------------------
[1] level:2:cluster:0001:cc1ab0a2eb
Univer

In [12]:
# Render the constructed tree with Graphviz.
# If the diagram is too dense, lower MAX_DEPTH or LABEL_CHARS.
from graphviz import Digraph


MAX_DEPTH = None
LABEL_CHARS = 120
GRAPH_DIR = REPO_ROOT / "results" / "tree_construction"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_PATH = GRAPH_DIR / "semantic_tree_graphviz"


def shorten(text, limit=LABEL_CHARS):
    text = str(text).replace("\n", " ").strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."


def add_tree_to_dot(dot, node, path=(), max_depth=None):
    node_name = "root" if path == () else "node_" + "_".join(map(str, path))
    children = node.get("child") or []
    desc = shorten(node.get("desc", ""))
    node_id = node.get("id")
    is_truncated = max_depth is not None and len(path) >= max_depth and len(children) > 0
    truncation_note = "\n[TRUNCATED]" if is_truncated else ""
    label = f"{node_name}\n{node_id}{truncation_note}\n{desc}" if node_id is not None else f"{node_name}{truncation_note}\n{desc}"

    fillcolor = "lightsalmon" if is_truncated else "lightgoldenrod1"
    dot.node(node_name, label=label, shape="box", style="rounded,filled", fillcolor=fillcolor)

    if max_depth is not None and len(path) >= max_depth:
        return

    for idx, child in enumerate(children):
        child_path = path + (idx,)
        child_name = "node_" + "_".join(map(str, child_path))
        add_tree_to_dot(dot, child, child_path, max_depth=max_depth)
        dot.edge(node_name, child_name)


dot = Digraph(comment="Semantic Tree")
dot.attr(rankdir="TB")
dot.attr("node", fontname="Helvetica", fontsize="10")
dot.attr("edge", color="gray50")

add_tree_to_dot(dot, tree_dict, max_depth=MAX_DEPTH)
png_path = dot.render(str(GRAPH_PATH), format="svg", cleanup=True)
print(f"Saved Graphviz PNG to {png_path}")
dot


Saved Graphviz PNG to C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\semantic_tree_graphviz.svg


In [9]:
def count_rendered_nodes(node, path=(), max_depth=None):
    total = 1
    if max_depth is not None and len(path) >= max_depth:
        return total
    for idx, child in enumerate(node.get("child") or []):
        total += count_rendered_nodes(child, path + (idx,), max_depth=max_depth)
    return total

def count_rendered_terminal_nodes(node, path=(), max_depth=None):
    children = node.get("child") or []
    if max_depth is not None and len(path) >= max_depth:
        return 1
    if not children:
        return 1
    total = 0
    for idx, child in enumerate(children):
        total += count_rendered_terminal_nodes(child, path + (idx,), max_depth=max_depth)
    return total

print("Rendered nodes:", count_rendered_nodes(tree_dict, max_depth=MAX_DEPTH))
print("Full nodes:", count_nodes(tree_dict))
print("Rendered terminal nodes:", count_rendered_terminal_nodes(tree_dict, max_depth=MAX_DEPTH))
print("Full leaves:", count_leaves(tree_dict))
if MAX_DEPTH is not None:
    print("Note: rendered terminal nodes may be higher than true leaves because truncated internal nodes are drawn as endpoints.")

Rendered nodes: 97
Full nodes: 97
Full leaves: 64


In [11]:
# Optional: persist the notebook-built tree so it can be used by the retrieval pipeline.
OUTPUT_DIR = REPO_ROOT / "trees" / args.dataset / "codigo_civil_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pickle_path = OUTPUT_DIR / "tree-bottom-up-llm.pkl"
json_path = OUTPUT_DIR / "tree-bottom-up-llm.json"

pickle_path.write_bytes(pickle.dumps(tree_dict))
json_path.write_text(json.dumps(tree_dict, ensure_ascii=False, indent=2), encoding="utf-8")

print(pickle_path)
print(json_path)


C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\PT\codigo_civil_notebook\tree-bottom-up-llm.pkl
C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\PT\codigo_civil_notebook\tree-bottom-up-llm.json
